In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from scipy.stats import norm
from numba import njit, prange, get_thread_id, get_num_threads

from astropy.time import Time
from astropy import units
from astropy.coordinates import SkyCoord, EarthLocation, solar_system_ephemeris
from astropy.utils import iers


# ============================================================
# 0) Obs Window Processing and Time Conversion
# ============================================================

def load_windows_txt(path_txt):
    with open(path_txt, 'r') as f:
        lines = f.read().splitlines()
    data_lines = [ln for ln in lines if (ln.strip() != "") and (not ln.strip().startswith('#'))]
    flat_numbers = ' '.join(data_lines).split()
    arr = np.array(flat_numbers, dtype=float).reshape(-1, 2)
    return arr

def topo_mjdutc_to_tcb_ssb_mjd(times_topo_mjdutc, obs_location, source_coord):
    times_topo_mjdutc = np.asarray(times_topo_mjdutc, dtype=float)
    t_utc = Time(times_topo_mjdutc, format='mjd', scale='utc', location=obs_location)
    ltt = t_utc.light_travel_time(source_coord, kind='barycentric')
    t_ssb_tcb = t_utc.tcb + ltt
    return t_ssb_tcb.mjd

def get_windows_tcb_mjd(win_txt, obs_location, source_coord):
    w_topo = load_windows_txt(win_txt)
    start_topo = w_topo[:, 0]
    end_topo = w_topo[:, 1]
    start_tcb = topo_mjdutc_to_tcb_ssb_mjd(start_topo, obs_location, source_coord)
    end_tcb = topo_mjdutc_to_tcb_ssb_mjd(end_topo, obs_location, source_coord)
    
    windows_tcb_mjd = np.column_stack((start_tcb, end_tcb))
    
    bad = windows_tcb_mjd[:, 1] <= windows_tcb_mjd[:, 0]
    if np.any(bad):
        print("Warning: some converted windows have end<=start.")
    return windows_tcb_mjd


# ============================================================
# 1) Data Reading and Preprocessing
# ============================================================

def Xreaddata(path, col_indices, new_col_names, startrow=0):
    if len(col_indices) != len(new_col_names):
        raise ValueError("Column indices length mismatch.")
    df_0 = pd.read_csv(path, dtype=str)
    df = df_0.iloc[startrow:, col_indices]
    df.columns = new_col_names
    for col in df.columns:
        try:
            df[col] = df[col].astype(float)
        except ValueError:
            pass
    df = df.fillna(np.nan)
    df = df.sort_values(by='t').reset_index(drop=True)
    return df_0, df

def Xselectdata2(df_s, nummin, dnum, sep):
    t_min = np.floor(df_s['t'].min())
    t_max = np.ceil(df_s['t'].max())
    t_bins = np.arange(t_min - (1 - sep), t_max + (1 - sep) + dnum, dnum)
    num, _ = np.histogram(df_s['t'], t_bins)
    date_s = np.where(num >= nummin)[0] * dnum

    df_list = []
    base_t = np.floor(df_s['t'].min())
    for date in date_s:
        start_d = date + base_t - (1 - sep)
        end_d = start_d + dnum
        mask = (df_s['t'] > start_d) & (df_s['t'] < end_d)
        df_temp = df_s.loc[mask, :].reset_index(drop=True)
        df_list.append(df_temp)
    return df_s, df_list

def burstverify_anyday(bi,nummin,dnum,sep) :
    bi_ss=Xselectdata2(bi[1],nummin=nummin,dnum=dnum,sep=sep)
    b_s=[]    
    for i in range(len(bi_ss[1])):
        b_s.append(bi_ss[1][i]['t']*86400)
        bi_ss[1][i]['t_s']=b_s[i]   
        waitingtime=np.diff(b_s[i])
        bi_ss[1][i]['waitingtime']=np.concatenate(([np.inf],waitingtime))    
    return bi_ss

def get_cluster_representatives(df_day, threshold):
    df = df_day.sort_values('t_s').reset_index(drop=True).copy()
    t = df['t_s'].to_numpy(np.float64)
    dt_prev = np.empty_like(t)
    dt_prev[0] = np.inf
    dt_prev[1:] = np.diff(t)

    n = len(df)

    cluster_id = np.empty(n, dtype=np.int32)
    cid = 0
    for i in range(n):
        if i == 0:
            cluster_id[i] = cid
        else:
            if dt_prev[i] < threshold:
                cluster_id[i] = cid
            else:
                cid += 1
                cluster_id[i] = cid
    df['cluster_id'] = cluster_id

    reps = []
    for _, g in df.groupby('cluster_id', sort=False):
        idx = g['s'].idxmax()
        rep = g.loc[idx].copy()
        reps.append(rep)
    return pd.DataFrame(reps).reset_index(drop=True)


# ============================================================
# 2) Simulated data generation (uniform null)
# ============================================================

def count_events_in_windows_inclusive(event_times_mjd, windows_mjd_2col):
    event_times_mjd = np.asarray(event_times_mjd, dtype=float)
    starts = windows_mjd_2col[:, 0]
    ends = windows_mjd_2col[:, 1]
    
    counts = np.empty(len(starts), dtype=int)
    for i in range(len(starts)):
        counts[i] = np.sum((event_times_mjd >= starts[i]) & (event_times_mjd <= ends[i]))
    return counts

def prepare_day_windows_and_counts_with_win(day_df_any, windows_tcb_mjd):
    """
    Return intervals_s, counts_sel, win_sel
    """
    event_times_mjd = day_df_any['t'].to_numpy(float)
    counts_all = count_events_in_windows_inclusive(event_times_mjd, windows_tcb_mjd)
    idx = np.where(counts_all > 0)[0]
    if idx.size == 0:
        return None, None, None

    win_sel = windows_tcb_mjd[idx]
    counts_sel = counts_all[idx].astype(int)
    intervals_s = [(float(a) * 86400.0, float(b) * 86400.0) for a, b in win_sel]
    return intervals_s, counts_sel, win_sel

def prepare_day_windows_and_counts(day_cluster_df, windows_tcb_mjd):
    intervals_s, counts_sel, _ = prepare_day_windows_and_counts_with_win(day_cluster_df, windows_tcb_mjd)
    return intervals_s, counts_sel

def generate_N_toaseries_with_windows_fixed_counts(N, intervals_s, counts_per_interval, rng=None):
    """
    null: Fixed window + fixed count; uniform sampling within the window and sorting
    """
    if rng is None:
        rng = np.random.default_rng()

    counts = np.asarray(counts_per_interval, dtype=int)
    starts = np.array([a for a, b in intervals_s], dtype=np.float64)
    ends   = np.array([b for a, b in intervals_s], dtype=np.float64)

    order = np.argsort(starts)
    starts, ends, counts = starts[order], ends[order], counts[order]

    N_total = int(np.sum(counts))
    out = np.empty((N, N_total), dtype=np.float64)

    col = 0
    for a, b, Nj in zip(starts, ends, counts):
        Nj = int(Nj)
        if Nj == 0:
            continue
        seg = rng.uniform(a, b, size=(N, Nj)).astype(np.float64)
        seg.sort(axis=1)
        out[:, col:col+Nj] = seg
        col += Nj
    return out

def build_s_template_from_windows(day_df, win_sel):
    t = day_df['t'].to_numpy(float)
    s = day_df['s'].to_numpy(float)

    s_list = []
    for (a, b) in win_sel:
        m = (t >= a) & (t <= b)
        s_list.append(s[m])

    if len(s_list) == 0:
        return np.empty(0, dtype=np.float64)
    return np.concatenate(s_list).astype(np.float64)


# ============================================================
# 2.5) MC brightest clustering
# ============================================================

def make_perm_matrix_fy(N_mc, B, rng, dtype=np.int32):
    perm = np.tile(np.arange(B, dtype=dtype), (N_mc, 1))
    rows = np.arange(N_mc, dtype=np.int64)
    tmp = np.empty(N_mc, dtype=dtype)
    for i in range(B - 1, 0, -1):
        j = rng.integers(0, i + 1, size=N_mc, dtype=np.int64)
        tmp[:] = perm[:, i]
        perm[:, i] = perm[rows, j]
        perm[rows, j] = tmp
    return perm

@njit(parallel=True, fastmath=True)
def cluster_reps_brightest_given_perm(toas_matrix, perm, s_template, threshold):
    N_mc, B = toas_matrix.shape
    reps = np.empty((N_mc, B), dtype=np.float64)
    nrep = np.empty(N_mc, dtype=np.int32)
    for s in prange(N_mc):
        if B == 0:
            nrep[s] = 0
            continue
        out_i = 0
        best_idx = 0
        best_s = float(s_template[perm[s, 0]])
        for i in range(1, B + 1):
            end_cluster = False
            if i == B:
                end_cluster = True
            else:
                dt = toas_matrix[s, i] - toas_matrix[s, i - 1]
                if dt >= threshold:
                    end_cluster = True
            if not end_cluster:
                sv = float(s_template[perm[s, i]])
                if sv > best_s:
                    best_s = sv
                    best_idx = i
            else:
                reps[s, out_i] = toas_matrix[s, best_idx]
                out_i += 1
                if i < B:
                    best_idx = i
                    best_s = float(s_template[perm[s, i]])
        nrep[s] = out_i
        for k in range(out_i, B):
            reps[s, k] = 0.0
    return reps, nrep


# ============================================================
# 3) Chi2 calculation
# ============================================================

@njit(parallel=True, fastmath=True)
def batch_find_max_chi2_fast_flex(toas_matrix, n_arr, p_arr, bins_num=30):
    N_sim, N_max = toas_matrix.shape
    M = p_arr.shape[0]
    n_threads = get_num_threads()

    bins_f = float(bins_num)
    bins_minus_1 = float(bins_num - 1)
    f_bins = bins_f

    inv_norm_s = np.zeros(N_sim, dtype=np.float64)
    nE_s = np.zeros(N_sim, dtype=np.float64)

    for s in range(N_sim):
        n = int(n_arr[s])
        if n >= 2:
            n_f = float(n)
            E = n_f / bins_f
            q = 1.0 + (bins_f + 1.0) / (6.0 * n_f)
            inv_norm_s[s] = 1.0 / (E * bins_minus_1 * q)
            nE_s[s] = n_f * E
        else:
            inv_norm_s[s] = 0.0
            nE_s[s] = 0.0

    tls_max_vals = np.full((n_threads, N_sim), -1.0, dtype=np.float64)
    tls_best_p   = np.zeros((n_threads, N_sim), dtype=np.float64)

    chunk_len = 4096
    n_chunks = (M + chunk_len - 1) // chunk_len
    UF = 4

    for c in prange(n_chunks):
        tid = get_thread_id()
        start_idx = c * chunk_len
        end_idx = min(start_idx + chunk_len, M)
        limit = start_idx + ((end_idx - start_idx) // UF) * UF

        multi_counts = np.zeros((UF, bins_num), dtype=np.int32)
        single_counts = np.zeros(bins_num, dtype=np.int32)

        for i in range(start_idx, limit, UF):
            f0, p0 = 1.0 / p_arr[i],   p_arr[i]
            f1, p1 = 1.0 / p_arr[i+1], p_arr[i+1]
            f2, p2 = 1.0 / p_arr[i+2], p_arr[i+2]
            f3, p3 = 1.0 / p_arr[i+3], p_arr[i+3]

            for s in range(N_sim):
                n = int(n_arr[s])
                if n < 2:
                    continue

                inv_norm = inv_norm_s[s]
                nE = nE_s[s]

                multi_counts[:] = 0
                for k in range(n):
                    val = toas_matrix[s, k]

                    ph = val * f0; ph -= np.floor(ph)
                    idx = int(ph * f_bins)
                    if idx >= bins_num: idx = bins_num - 1
                    multi_counts[0, idx] += 1

                    ph = val * f1; ph -= np.floor(ph)
                    idx = int(ph * f_bins)
                    if idx >= bins_num: idx = bins_num - 1
                    multi_counts[1, idx] += 1

                    ph = val * f2; ph -= np.floor(ph)
                    idx = int(ph * f_bins)
                    if idx >= bins_num: idx = bins_num - 1
                    multi_counts[2, idx] += 1

                    ph = val * f3; ph -= np.floor(ph)
                    idx = int(ph * f_bins)
                    if idx >= bins_num: idx = bins_num - 1
                    multi_counts[3, idx] += 1

                curr_max = tls_max_vals[tid, s]
                curr_best = tls_best_p[tid, s]

                s2 = 0.0
                for b in range(bins_num):
                    x = float(multi_counts[0, b]); s2 += x*x
                chi = (s2 - nE) * inv_norm
                if chi > curr_max:
                    curr_max = chi; curr_best = p0

                s2 = 0.0
                for b in range(bins_num):
                    x = float(multi_counts[1, b]); s2 += x*x
                chi = (s2 - nE) * inv_norm
                if chi > curr_max:
                    curr_max = chi; curr_best = p1

                s2 = 0.0
                for b in range(bins_num):
                    x = float(multi_counts[2, b]); s2 += x*x
                chi = (s2 - nE) * inv_norm
                if chi > curr_max:
                    curr_max = chi; curr_best = p2

                s2 = 0.0
                for b in range(bins_num):
                    x = float(multi_counts[3, b]); s2 += x*x
                chi = (s2 - nE) * inv_norm
                if chi > curr_max:
                    curr_max = chi; curr_best = p3

                tls_max_vals[tid, s] = curr_max
                tls_best_p[tid, s] = curr_best

        for i in range(limit, end_idx):
            f, p_val = 1.0 / p_arr[i], p_arr[i]
            for s in range(N_sim):
                n = int(n_arr[s])
                if n < 2:
                    continue

                inv_norm = inv_norm_s[s]
                nE = nE_s[s]

                single_counts[:] = 0
                for k in range(n):
                    val = toas_matrix[s, k]
                    ph = val * f; ph -= np.floor(ph)
                    idx = int(ph * f_bins)
                    if idx >= bins_num: idx = bins_num - 1
                    single_counts[idx] += 1

                s2 = 0.0
                for b in range(bins_num):
                    x = float(single_counts[b]); s2 += x*x
                chi = (s2 - nE) * inv_norm

                if chi > tls_max_vals[tid, s]:
                    tls_max_vals[tid, s] = chi
                    tls_best_p[tid, s] = p_val

    final_max_vals = np.empty(N_sim, dtype=np.float32)
    final_best_p = np.empty(N_sim, dtype=np.float64)

    for s in range(N_sim):
        g_max = -1.0
        g_p = 0.0
        for t in range(n_threads):
            if tls_max_vals[t, s] > g_max:
                g_max = tls_max_vals[t, s]
                g_p = tls_best_p[t, s]
        final_max_vals[s] = g_max
        final_best_p[s] = g_p

    return final_max_vals, final_best_p


# ============================================================
# Joint_significance_pair_mc
# ============================================================
def _simulate_one_day_chunked(
    *,
    day_idx: int,
    intervals_s,
    counts_sel: np.ndarray,
    s_template: np.ndarray,
    p_short: np.ndarray,
    p_long: np.ndarray,
    bins_num: int,
    cluster_threshold: float,
    N_pairs: int,
    sim_chunk_size: int,
    seed: int,
    chi_out: np.ndarray,
    P_out: np.ndarray,
    meanT_out: np.ndarray,
):


    rng_toa  = np.random.default_rng((seed, day_idx, 0))
    rng_perm = np.random.default_rng((seed, day_idx, 1))

    B = int(np.sum(counts_sel))
    if B != int(s_template.size):
        raise ValueError(f"Day{day_idx}: B(sum(counts))={B} != len(s_template)={s_template.size}")

    pbar = tqdm.tqdm(total=N_pairs, desc=f"Sims Day{day_idx} (chunked)", leave=True)

    pos = 0
    while pos < N_pairs:
        n_blk = min(sim_chunk_size, N_pairs - pos)

        # 1) raw TOA block
        mc = generate_N_toaseries_with_windows_fixed_counts(n_blk, intervals_s, counts_sel, rng=rng_toa)
        meanT_out[pos:pos+n_blk] = np.mean(mc, axis=1)

        # 2) short period
        if p_short.size > 0:
            n_raw = np.full(n_blk, mc.shape[1], dtype=np.int32)
            chi_short, P_short = batch_find_max_chi2_fast_flex(mc, n_raw, p_short, bins_num)
        else:
            chi_short = np.full(n_blk, -1.0, dtype=np.float32)
            P_short   = np.zeros(n_blk, dtype=np.float64)

        # 3) long period
        if p_long.size > 0:
            perm = make_perm_matrix_fy(n_blk, mc.shape[1], rng_perm, dtype=np.int32)
            reps, nrep = cluster_reps_brightest_given_perm(mc, perm, s_template, float(cluster_threshold))
            del perm

            chi_long, P_long = batch_find_max_chi2_fast_flex(reps, nrep, p_long, bins_num)
            del reps, nrep
        else:
            chi_long = np.full(n_blk, -1.0, dtype=np.float32)
            P_long   = np.zeros(n_blk, dtype=np.float64)

        # 4) combine（max over segments）
        chi_blk = np.maximum(chi_short, chi_long).astype(np.float32)
        mask = chi_short >= chi_long
        P_blk = np.where(mask, P_short, P_long).astype(np.float64)

        chi_out[pos:pos+n_blk] = chi_blk
        P_out[pos:pos+n_blk]   = P_blk

        del mc, chi_short, chi_long, P_short, P_long, chi_blk, P_blk

        pos += n_blk
        pbar.update(n_blk)

    pbar.close()

def joint_significance_pair_mc(
    b_ss,
    windows_tcb_mjd,
    day_idx_1, day_idx_2,
    pstart, pend, step,
    bins_num=30,
    cluster_threshold=0.5,
    N_pairs=100000,
    seed=42,
):
    print(f"Set up: Day {day_idx_1} vs {day_idx_2}, N_pairs={N_pairs}.")

    p_arr = np.linspace(pstart, pend, int((pend - pstart) / step) + 1, dtype=np.float64)
    print(f"Period grid: {len(p_arr):,} points.")

    split = float(cluster_threshold) * 2.0
    p_short = p_arr[p_arr < split]
    p_long  = p_arr[p_arr >= split]

    # -------------------------
    # obs：raw & reps
    # -------------------------
    day_df_1 = b_ss[1][day_idx_1]
    day_df_2 = b_ss[1][day_idx_2]

    t1_all = day_df_1['t_s'].to_numpy(np.float64)
    t2_all = day_df_2['t_s'].to_numpy(np.float64)

    t1_rep = get_cluster_representatives(day_df_1, threshold=cluster_threshold)['t_s'].to_numpy(np.float64)
    t2_rep = get_cluster_representatives(day_df_2, threshold=cluster_threshold)['t_s'].to_numpy(np.float64)

    meanT1_obs = float(np.mean(t1_all)) if t1_all.size > 0 else np.nan
    meanT2_obs = float(np.mean(t2_all)) if t2_all.size > 0 else np.nan
    Nburst1_obs = int(t1_all.size)
    Nburst2_obs = int(t2_all.size)

    def _max_obs(t, p_seg):
        if p_seg.size == 0 or t.size < 2:
            return -1.0, 0.0
        n = np.array([t.size], dtype=np.int32)
        chi, pp = batch_find_max_chi2_fast_flex(t.reshape(1, -1), n, p_seg, bins_num)
        return float(chi[0]), float(pp[0])

    print("Calculating Obs Chi2 (segmented, max over all segments).")

    chi1_short, P1_short = _max_obs(t1_all, p_short)
    chi1_long,  P1_long  = _max_obs(t1_rep, p_long)
    if chi1_short >= chi1_long:
        chi1_obs, P1_obs = chi1_short, P1_short
    else:
        chi1_obs, P1_obs = chi1_long, P1_long

    chi2_short, P2_short = _max_obs(t2_all, p_short)
    chi2_long,  P2_long  = _max_obs(t2_rep, p_long)
    if chi2_short >= chi2_long:
        chi2_obs, P2_obs = chi2_short, P2_short
    else:
        chi2_obs, P2_obs = chi2_long, P2_long

    dt_obs = meanT2_obs - meanT1_obs
    pdot_obs = (P2_obs - P1_obs) / dt_obs if (dt_obs != 0 and np.isfinite(dt_obs)) else np.nan

    print(f"Observed(seg): Chi1={chi1_obs:.2f}@{P1_obs:.6f}s, Chi2={chi2_obs:.2f}@{P2_obs:.6f}s")

    # -------------------------
    # Prepare windows/counts and brightness templates
    # -------------------------
    int1, cnt1, win1 = prepare_day_windows_and_counts_with_win(day_df_1, windows_tcb_mjd)
    int2, cnt2, win2 = prepare_day_windows_and_counts_with_win(day_df_2, windows_tcb_mjd)
    if int1 is None or int2 is None:
        raise RuntimeError("No valid windows with events. Check time system consistency and windows.")

    s1_template = build_s_template_from_windows(day_df_1, win1).astype(np.float64)
    s2_template = build_s_template_from_windows(day_df_2, win2).astype(np.float64)


    sim_chunk_size = max(1, N_pairs // 10)
    if sim_chunk_size > N_pairs:
        sim_chunk_size = N_pairs

    # Output array
    chi1_sims = np.empty(N_pairs, dtype=np.float32)
    P1_sims   = np.empty(N_pairs, dtype=np.float64)
    meanT1_sims = np.empty(N_pairs, dtype=np.float64)

    chi2_sims = np.empty(N_pairs, dtype=np.float32)
    P2_sims   = np.empty(N_pairs, dtype=np.float64)
    meanT2_sims = np.empty(N_pairs, dtype=np.float64)

    # Day1 / Day2 run
    _simulate_one_day_chunked(
        day_idx=day_idx_1,
        intervals_s=int1,
        counts_sel=cnt1,
        s_template=s1_template,
        p_short=p_short, p_long=p_long,
        bins_num=bins_num,
        cluster_threshold=cluster_threshold,
        N_pairs=N_pairs,
        sim_chunk_size=sim_chunk_size,
        seed=seed,
        chi_out=chi1_sims,
        P_out=P1_sims,
        meanT_out=meanT1_sims,
    )

    _simulate_one_day_chunked(
        day_idx=day_idx_2,
        intervals_s=int2,
        counts_sel=cnt2,
        s_template=s2_template,
        p_short=p_short, p_long=p_long,
        bins_num=bins_num,
        cluster_threshold=cluster_threshold,
        N_pairs=N_pairs,
        sim_chunk_size=sim_chunk_size,
        seed=seed,
        chi_out=chi2_sims,
        P_out=P2_sims,
        meanT_out=meanT2_sims,
    )

    # -------------------------
    # Significance
    # -------------------------
    p1 = float(np.mean(chi1_sims >= chi1_obs))
    p2 = float(np.mean(chi2_sims >= chi2_obs))
    sig1 = float(norm.isf(p1)) if p1 > 0 else float("inf")
    sig2 = float(norm.isf(p2)) if p2 > 0 else float("inf")

    meta = {
        "obs": {
            "chi1_obs": float(chi1_obs),
            "chi2_obs": float(chi2_obs),
            "P1_obs": float(P1_obs),
            "P2_obs": float(P2_obs),
            "meanT1_obs": float(meanT1_obs),
            "meanT2_obs": float(meanT2_obs),
            "pdot_obs": float(pdot_obs),
            "Nburst1": int(Nburst1_obs),
            "Nburst2": int(Nburst2_obs)
        },
        "config": {
            "days": [int(day_idx_1), int(day_idx_2)],
            "bins_num": int(bins_num),
            "cluster_threshold": float(cluster_threshold),
            "cluster_mode": "brightest",
            "N_pairs": int(N_pairs),
            "seed": int(seed),
            "p_start": float(pstart),
            "p_end": float(pend),
            "p_step": float(step),
            "M_period_grid": int(len(p_arr))
        },
        "results": {
            "p_day1": p1,
            "sigma_day1": sig1,
            "p_day2": p2,
            "sigma_day2": sig2
        }
    }

    arr = {
        "chi1_sims": chi1_sims,
        "chi2_sims": chi2_sims,
        "P1_sims": P1_sims,
        "P2_sims": P2_sims,
        "meanT1_sims": meanT1_sims,
        "meanT2_sims": meanT2_sims,
    }
    return meta, arr



# ============================================================
# Save
# ============================================================

def build_filename_from_meta(meta, prefix="PAIRCHI2x"):
    cfg = meta["config"]
    ps = cfg["p_start"]
    pe = cfg["p_end"]
    step = cfg["p_step"]
    th = cfg["cluster_threshold"]
    mode = cfg["cluster_mode"]
    seed = cfg["seed"]
    b = cfg["bins_num"]
    Np = cfg["N_pairs"]
    d1, d2 = cfg["days"]

    step_str = f"{step:.0e}".replace("+0", "").replace("+", "")
    th_str = f"{th:g}"

    filename = (
        f"{prefix}_seed{seed}_b{b}_N{Np}"
        f"_ps{ps:g}_pe{pe:g}_step{step_str}"
        f"_th{th_str}_{mode}_days{d1}-{d2}.npz"
    )
    return filename

def save_pair_mc_with_meta_filename(pf, filename, meta, arr):
    os.makedirs(pf, exist_ok=True)
    fullpath = os.path.join(pf, filename)

    meta_json_str = json.dumps(meta, ensure_ascii=False)
    np.savez_compressed(fullpath, **arr, meta_json=meta_json_str)

    json_path = fullpath.replace(".npz", ".json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print("Saved:")
    print("  NPZ :", fullpath)
    print("  JSON:", json_path)
    return fullpath


# ============================================================
# main
# ============================================================

if __name__ == "__main__":
    fast1_path = r"Data\20201124A\Burst_Table\FAST#1.csv"
    win_txt = r"Data\20201124A\FRB20201124A_fast1_obswindow_UTC.txt"
    
    iers.conf.auto_download = True
    solar_system_ephemeris.set('de440s')
    loc = EarthLocation(lon='106d51m24.0s', lat='25d39m10.6s', height=1110.0288 * units.m)
    coord = SkyCoord(ra='5h08m03.51s', dec='+26d03m38.50s', frame='icrs')
    
    w_tcb = get_windows_tcb_mjd(win_txt, loc, coord)
    b_df = Xreaddata(fast1_path, [0,2,6,4], ['t','s','w','f'])
    b_ss = burstverify_anyday(b_df, 1, 1, 0)
    
    start_time = time.time()
    meta, arr = joint_significance_pair_mc(
        b_ss, w_tcb, 3, 35,
        pstart=0.1, pend=100.0, step=1e-5,
        N_pairs=10, # if set to 1000000 the runtime would be ~ 10 hr
        bins_num=20,
        cluster_threshold=0.5,
        seed=2025
    )
    end_time = time.time()

    print("Runtime：", end_time - start_time, "s")  
    filename = build_filename_from_meta(meta)
    pf = r"MC Samples_Main\\"
    _ = save_pair_mc_with_meta_filename(pf, filename, meta, arr)

Set up: Day 3 vs 35, N_pairs=10.
Period grid: 9,990,001 points.
Calculating Obs Chi2 (segmented, max over all segments).
Observed(seg): Chi1=6.05@1.706030s, Chi2=5.28@1.707970s


Sims Day35 (chunked): 100%|████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 10.68it/s]

Runtime： 7.415917634963989 s
Saved:
  NPZ : MC Samples_Main\\PAIRCHI2x_seed2025_b20_N10_ps0.1_pe100_step1e-05_th0.5_brightest_days3-35.npz
  JSON: MC Samples_Main\\PAIRCHI2x_seed2025_b20_N10_ps0.1_pe100_step1e-05_th0.5_brightest_days3-35.json
